# 第10回　回帰分析：単回帰と重回帰
## ―― データを「数式（モデル）」で説明する

統計学Ⅰ（B）　／　北星学園大学

注目は ――

> モデルは現実の**近似**にすぎない。**当てはまりの良さ（R²）は、良いモデルの証明ではない。**

### フック

> 「**通学時間**が分かれば、その人の**テスト点**を予測できるだろうか？」
>
> 散布図に1本の直線を引いて、予測してみよう。うまくいくか？

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
import statsmodels.api as sm

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan
df = df.dropna(subset=["睡眠時間h","通学時間min"])
print("準備OK")

In [ ]:
# 通学時間でテスト点を予測してみる（単回帰）
y = df["テスト点"]
X = sm.add_constant(df[["通学時間min"]])
m0 = sm.OLS(y, X).fit()
print(f"傾き = {m0.params['通学時間min']:.3f}　R² = {m0.rsquared:.3f}")
print("→ R²はほぼ0。通学時間ではテスト点をほとんど説明できない。")

通学時間では予測できなかった（R²≈0、直線はほぼ水平）。では**勉強時間**ならどうか？

In [ ]:
X = sm.add_constant(df[["勉強時間h"]])
m1 = sm.OLS(y, X).fit()
切片 = m1.params["const"]; 傾き = m1.params["勉強時間h"]
print(f"テスト点 ≈ {切片:.1f} + {傾き:.1f} × 勉強時間")
print(f"R² = {m1.rsquared:.3f}（テスト点のばらつきの約{m1.rsquared*100:.0f}%を説明）")

plt.figure(figsize=(6,4))
plt.scatter(df["勉強時間h"], y, s=12, alpha=0.4, color="#00897b")
xs = np.linspace(df["勉強時間h"].min(), df["勉強時間h"].max(), 50)
plt.plot(xs, 切片 + 傾き*xs, color="#e8503a", lw=2, label="回帰直線")
plt.xlabel("勉強時間h"); plt.ylabel("テスト点"); plt.legend()
plt.title(f"単回帰：勉強1時間で約+{傾き:.1f}点（予測）"); plt.show()

---
## 1. 回帰の言葉

- **回帰直線**：データに最もよく当てはまる直線（最小二乗法：縦のズレの二乗和が最小）
- **傾き（回帰係数）**：説明変数が1増えると、予測値がいくつ増えるか。今回「勉強1時間で約+7.5点」
- **切片**：説明変数が0のときの予測値
- **決定係数 R²**（0〜1）：目的変数のばらつきのうち、モデルが説明できた割合。1に近いほどよく当てはまる

> ⚠️ 「勉強1時間で+7.5点」は**予測上の関連**であって、「勉強すれば必ず7.5点上がる」という**因果の保証ではない**（第5回）。

---
## 2. 重回帰 ―― 複数の説明変数で

勉強だけでなく、睡眠・SNSも一緒に使う。重回帰では各係数が「**他の変数を一定にしたときの**、その変数の関連」を表す（＝統制）。

In [ ]:
X = sm.add_constant(df[["勉強時間h", "睡眠時間h", "SNS時間h"]])
m2 = sm.OLS(y, X).fit()
print("テスト点 ≈ {:.1f} + {:.2f}×勉強 + {:.2f}×睡眠 + {:.2f}×SNS".format(
    m2.params['const'], m2.params['勉強時間h'], m2.params['睡眠時間h'], m2.params['SNS時間h']))
print(f"R² = {m2.rsquared:.3f}")
print("\n係数の符号：勉強(+)・睡眠(+)・SNS(−)。SNSが多いほど予測点は下がる。")
print("各係数は『他を一定にしたとき』の関連。SNSの−0.76 は、勉強・睡眠が同じ人どうしで")
print("比べたときの、SNS1時間あたりの予測点の差。")

---
## 3. R²の罠 ―― 変数を足せば、必ず上がる

ここが今日の核心。**説明変数を足すと、R²は必ず上がる（決して下がらない）。** たとえそれが**まったく無意味な数字**でも。確かめよう：テスト点とは何の関係もない**でたらめな乱数の列**を足していく。

In [ ]:
rng = np.random.default_rng(2026)
base = df[["勉強時間h"]].copy()
print("『勉強時間』に、テスト点と無関係な乱数列を足していくと R² は…")
for k in [0, 1, 5, 20]:
    X = base.copy()
    for j in range(k):
        X[f"でたらめ{j}"] = rng.normal(0, 1, len(df))   # 意味のない数字
    r2 = sm.OLS(y, sm.add_constant(X)).fit().rsquared
    print(f"  でたらめ列 {k:2d}本追加: R² = {r2:.3f}")
print("\n中身が無意味でも、変数を足すほど R² は上がる。")
print("→ 『R²が高い＝良いモデル』ではない！ 変数を増やせばいくらでも盛れる。")

でたらめな列を20本足すだけで、R²は 0.36 → 0.41 に上がった。中身はゴミなのに「当てはまり」は良くなる。これが **過学習** の入り口だ。

> 当てはまり（R²）を上げたいだけなら、変数をたくさん入れればいい。だが、それは**手元のデータに合わせすぎた**だけで、**未来の予測は良くならない**。
> 「当てはまりの良さ」と「モデルの複雑さ」をどう釣り合わせるか ―― それが**次回（第11回 AIC）**のテーマ。

---
## 4. 残差を見る習慣 ／ 外挿の危険

**残差**＝実際の値 − 予測値。残差を予測値に対してプロットし、偏った模様がないか確認するのが作法（ランダムに散っていれば良い兆候）。

In [ ]:
plt.figure(figsize=(6,3.5))
plt.scatter(m1.fittedvalues, m1.resid, s=12, alpha=0.4, color="#00897b")
plt.axhline(0, color="#e8503a", lw=2)
plt.xlabel("予測値"); plt.ylabel("残差（実際−予測）"); plt.title("残差プロット：偏りなく散っていればOK")
plt.show()

# 外挿の危険：データ範囲（勉強0〜約4h）の外を予測すると…
print(f"勉強時間の実際の範囲: {df['勉強時間h'].min():.1f}〜{df['勉強時間h'].max():.1f} 時間")
print(f"もし勉強20時間/日 を式に入れると予測点 = {切片 + 傾き*20:.0f}点（100超え！＝あり得ない）")
print("→ モデルはデータのある範囲でしか信用できない。範囲外の外挿は危険。")

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 単回帰 | y ≈ 切片 + 傾き×x。傾き＝xが1増えると予測がいくつ増えるか |
| 決定係数 R² | ばらつきのうちモデルが説明できた割合（0〜1） |
| 重回帰 | 複数の説明変数。各係数は「他を一定にしたとき」の関連（統制） |
| ❌ R²の罠 | 変数を足せばR²は必ず上がる。R²高い≠良いモデル |
| ❌ 係数の罠 | 回帰係数＝予測上の関連。因果の保証ではない |
| ❌ 外挿の罠 | データ範囲の外は予測できない（勉強20h→点100超） |
| 残差 | 実際−予測。偏りがないか必ず見る |

> **モデルは現実の近似。「当てはまりの良さ」は「良いモデル」ではない。**
> 係数を因果と読まず、範囲外を予測せず、残差を見る。

**課題（Moodle）**：回帰結果を解釈し、「この予測の限界・使ってはいけない場面」を述べる。